# 🤖 PCB Routing Optimization (RL-style)

Goal:
- Optimize routing paths
- Minimize path length / congestion
- Experiment with RL-like search strategies

This is NOT full RL yet — but structured for it.

In [ ]:
# Setup
from generation.routing.autorouter import autoroute
from generation.routing.graph_router import graph_route
from generation.render.pcb_plot import draw_pcb

import copy
import random
import pprint

## 🧩 Sample Design

In [ ]:
design = {
    "components": [
        {"ref": "U1"},
        {"ref": "R1"},
        {"ref": "C1"},
        {"ref": "R2"},
    ],
    "layout": {
        "U1": {"x": 0, "y": 0},
        "R1": {"x": 20, "y": 20},
        "C1": {"x": 40, "y": 40},
        "R2": {"x": 60, "y": 20},
    },
    "nets": [
        {"name": "SIG", "connections": ["U1:1", "R1:1", "C1:1"]},
        {"name": "PWR", "connections": ["U1:2", "R2:1"]},
    ],
    "routes": []
}

## 🧪 Baseline Routing

In [ ]:
baseline = autoroute(copy.deepcopy(design))

pprint.pprint(baseline["routes"])
draw_pcb(baseline)

## 🎯 Cost Function (Reward)

In [ ]:
def routing_cost(routes):
    """
    Lower is better
    """
    total_length = 0
    bends = 0

    for r in routes:
        path = r.get("path", [])

        total_length += len(path)

        # count bends
        for i in range(1, len(path) - 1):
            if path[i-1][0] != path[i][0] and path[i][1] != path[i+1][1]:
                bends += 1

    return total_length + bends * 2

## 🔍 Evaluate Baseline

In [ ]:
baseline_cost = routing_cost(baseline["routes"])
print("Baseline Cost:", baseline_cost)

## 🔁 Randomized Exploration (Pseudo-RL)

In [ ]:
best = None
best_cost = float("inf")

for i in range(10):
    candidate = graph_route(copy.deepcopy(design))
    cost = routing_cost(candidate["routes"])

    print(f"Iteration {i} Cost:", cost)

    if cost < best_cost:
        best = candidate
        best_cost = cost

print("\nBest Cost:", best_cost)
draw_pcb(best)

## 🧠 Exploration vs Exploitation

In [ ]:
def explore_or_exploit(prob=0.3):
    return random.random() < prob

best = None
best_cost = float("inf")

for i in range(20):
    if explore_or_exploit():
        candidate = graph_route(copy.deepcopy(design))
    else:
        candidate = autoroute(copy.deepcopy(design))

    cost = routing_cost(candidate["routes"])

    if cost < best_cost:
        best = candidate
        best_cost = cost

print("Best Hybrid Cost:", best_cost)
draw_pcb(best)

## 📊 Compare Results

In [ ]:
print("Baseline:", baseline_cost)
print("Optimized:", best_cost)

## 🚀 Future Work

- True RL (Q-learning / PPO)
- State = grid + obstacles
- Action = next routing step
- Reward = length + violations
- Multi-agent routing
- GPU acceleration